# Notebook 02 — Collect Raw GDELT Labor-Market News

## Goal
Collect raw news article metadata from the GDELT Document 2.0 API for six labor-market query groups.
Save the **untouched raw** news file to Google Drive. No cleaning happens here.

**We collect only metadata:** title, URL, date, domain, language, query group.  
**We do NOT scrape full article text.**

Query groups: `layoffs`, `hiring`, `unemployment`, `job_openings`, `wages`, `uncertainty`

---

## What can go wrong
- GDELT API is rate-limited; be patient with large date ranges
- GDELT coverage is uneven before 2014 (less full-text indexing)
- Some months may return 0 results for niche queries
- Duplicate articles are expected (syndication); cleaning happens in Notebook 04
- GDELT response format may vary; check raw JSON before saving

---

## What you must inspect
- Is total volume reasonable? (expect 50–500 articles/month for active query groups)
- Do sample titles look relevant to US labor markets?
- Are there months with 0 articles? (suspicious if not COVID or early 2000s)
- Are source domains credible news sites?
- Are duplicate URL counts manageable?

In [ ]:
import os
import time
import json
import pathlib
import datetime

import pandas as pd
import numpy as np
import requests
import yaml
import matplotlib.pyplot as plt
from tqdm import tqdm

DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/labor_news_rag_project')
REPO_DIR = pathlib.Path('/content/economic-news-labor-rag')

RAW_GDELT_DIR = DRIVE_ROOT / 'data' / 'raw' / 'gdelt'
DQ_DIR = DRIVE_ROOT / 'outputs' / 'data_quality'
APPROVALS_DIR = DRIVE_ROOT / 'approvals'

with open(REPO_DIR / 'configs' / 'gdelt_queries.yaml') as f:
    gdelt_cfg = yaml.safe_load(f)

with open(REPO_DIR / 'configs' / 'base.yaml') as f:
    base_cfg = yaml.safe_load(f)

_s = base_cfg['test_sample'] if base_cfg.get('test_mode', False) else base_cfg['sample']
START_DATE = pd.Timestamp(_s['start_date'])
END_DATE = pd.Timestamp(_s['end_date'])

MODE = 'TEST (quick run)' if base_cfg.get('test_mode', False) else 'FULL (research run)'
print(f'Mode       : {MODE}')
print(f'Window     : {START_DATE.date()} to {END_DATE.date()}')
print(f'Query groups: {list(gdelt_cfg["query_groups"].keys())}')
print()
print('To switch modes, set test_mode: true/false in configs/base.yaml')

## GDELT API collection function

In [ ]:
def fetch_gdelt_month(keyword, start_dt, end_dt, max_records=250, sleep_s=2):
    """
    Query GDELT DOC 2.0 API for one keyword and one time window.
    Returns a list of article dicts or empty list on failure.
    """
    url = 'https://api.gdeltproject.org/api/v2/doc/doc'
    query = f'({keyword}) sourcelang:english sourcecountry:US'
    params = {
        'query': query,
        'mode': 'artlist',
        'maxrecords': max_records,
        'startdatetime': start_dt.strftime('%Y%m%d%H%M%S'),
        'enddatetime': end_dt.strftime('%Y%m%d%H%M%S'),
        'format': 'json',
        'sort': 'DateDesc',
    }
    for attempt in range(3):
        try:
            r = requests.get(url, params=params, timeout=30)
            r.raise_for_status()
            data = r.json()
            return data.get('articles', [])
        except Exception as e:
            print(f'  Attempt {attempt+1} failed for {keyword}: {e}')
            time.sleep(sleep_s * (attempt + 1))
    return []


def collect_query_group(group_name, group_cfg, start_date, end_date, sleep_s=2):
    """
    Collect one query group month by month over the full date range.
    Uses the first keyword as the primary query term.
    """
    keywords = group_cfg['keywords']
    primary_keyword = ' OR '.join(f'"{kw}"' for kw in keywords[:3])  # use top 3 keywords

    months = pd.date_range(start_date, end_date, freq='MS')
    all_articles = []

    for month_start in tqdm(months, desc=f'{group_name}', leave=False):
        month_end = month_start + pd.offsets.MonthEnd(0)
        month_end = month_end.replace(hour=23, minute=59, second=59)

        articles = fetch_gdelt_month(
            primary_keyword,
            month_start,
            month_end,
            max_records=gdelt_cfg['api']['max_records'],
            sleep_s=sleep_s,
        )

        for art in articles:
            art['query_group'] = group_name
            art['collection_month'] = month_start.strftime('%Y-%m')

        all_articles.extend(articles)
        time.sleep(sleep_s)

    return all_articles

## Collect all query groups

**This cell may take 30–90 minutes** for the full 2000–2024 date range.

To run a quick test instead, set `test_mode: true` in `configs/base.yaml` — the setup cell above will pick 2022–2024 automatically.

In [ ]:
all_articles = []
for group_name, group_cfg in tqdm(gdelt_cfg['query_groups'].items(), desc='Query groups'):
    articles = collect_query_group(group_name, group_cfg, START_DATE, END_DATE, sleep_s=2)
    all_articles.extend(articles)
    print(f'  {group_name}: {len(articles)} articles')

print(f'\nTotal articles collected: {len(all_articles)}')

In [ ]:
news_df = pd.DataFrame(all_articles)

if len(news_df) == 0:
    raise RuntimeError('STOP: No articles collected. Check GDELT API connectivity and query config.')

# Standardize column names
rename_map = {
    'url': 'url',
    'title': 'title',
    'seendate': 'seendate',
    'domain': 'domain',
    'language': 'language',
    'sourcecountry': 'sourcecountry',
    'socialimage': 'socialimage',
}
news_df = news_df.rename(columns={k: v for k, v in rename_map.items() if k in news_df.columns})

# Parse date
if 'seendate' in news_df.columns:
    news_df['seendate'] = pd.to_datetime(news_df['seendate'], format='%Y%m%dT%H%M%SZ', errors='coerce')

print(f'news_df shape: {news_df.shape}')
print(f'Columns: {list(news_df.columns)}')
print(news_df.head(3))

## Diagnostic tables and plots

In [ ]:
print('=== Total rows:', len(news_df))
print()

print('=== Rows by query group ===')
print(news_df['query_group'].value_counts().to_string())
print()

print('=== Language distribution ===')
print(news_df['language'].value_counts().head(10).to_string())
print()

print('=== Missing title count:', news_df['title'].isna().sum())
print('=== Missing URL count:', news_df['url'].isna().sum())
print('=== Duplicate URL count:', news_df['url'].duplicated().sum())
print()

if 'domain' in news_df.columns:
    print('=== Top 20 source domains ===')
    print(news_df['domain'].value_counts().head(20).to_string())

In [ ]:
if 'seendate' in news_df.columns:
    news_df['article_month'] = news_df['seendate'].dt.to_period('M').astype(str)
    monthly = news_df.groupby('article_month').size().reset_index(name='article_count')

    print('=== Monthly article counts (first 12) ===')
    print(monthly.head(12).to_string(index=False))
    print()
    print(f'Months with 0 articles: {(monthly["article_count"] == 0).sum()}')
    print(f'Min articles in a month: {monthly["article_count"].min()}')
    print(f'Max articles in a month: {monthly["article_count"].max()}')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Plot 1: monthly volume
if 'article_month' in news_df.columns:
    monthly_vol = news_df.groupby('article_month').size()
    axes[0].plot(range(len(monthly_vol)), monthly_vol.values)
    axes[0].set_title('Monthly Article Volume (all query groups)')
    axes[0].set_xlabel('Month index')
    axes[0].set_ylabel('Article count')
    axes[0].grid(True, alpha=0.3)

# Plot 2: articles by query group over time
if 'article_month' in news_df.columns:
    pivot = news_df.groupby(['article_month', 'query_group']).size().unstack(fill_value=0)
    for col in pivot.columns:
        axes[1].plot(range(len(pivot)), pivot[col].values, label=col, alpha=0.8)
    axes[1].set_title('Article Count by Query Group Over Time')
    axes[1].set_xlabel('Month index')
    axes[1].set_ylabel('Count')
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)

# Plot 3: top domains
if 'domain' in news_df.columns:
    top_domains = news_df['domain'].value_counts().head(20)
    axes[2].barh(top_domains.index[::-1], top_domains.values[::-1])
    axes[2].set_title('Top 20 Source Domains')
    axes[2].set_xlabel('Article count')
    axes[2].tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.savefig(str(DRIVE_ROOT / 'outputs' / 'figures' / 'gdelt_collection_overview.png'), dpi=100)
plt.show()
print('Plot saved.')

In [ ]:
def show_random_titles(df, query_group, n=10):
    sub = df[df['query_group'] == query_group].dropna(subset=['title'])
    sample = sub.sample(min(n, len(sub)), random_state=42)['title'].tolist()
    print(f'\n=== Sample titles for [{query_group}] (n={len(sample)}) ===')
    for i, t in enumerate(sample, 1):
        print(f'  {i:2d}. {t}')

for grp in gdelt_cfg['query_groups'].keys():
    show_random_titles(news_df, grp, n=10)

## Save raw news file (untouched)

In [ ]:
raw_path = RAW_GDELT_DIR / 'news_raw.parquet'
news_df.to_parquet(raw_path, index=False)
print(f'Raw news data saved: {raw_path}')
print(f'Shape: {news_df.shape}')
print(f'File size: {raw_path.stat().st_size / 1024:.1f} KB')

# Save data quality report
dq_rows = []
for grp in gdelt_cfg['query_groups'].keys():
    sub = news_df[news_df['query_group'] == grp]
    dq_rows.append({
        'query_group': grp,
        'total_articles': len(sub),
        'missing_title': sub['title'].isna().sum(),
        'missing_url': sub['url'].isna().sum() if 'url' in sub.columns else 0,
        'duplicate_url': sub['url'].duplicated().sum() if 'url' in sub.columns else 0,
    })
dq_df = pd.DataFrame(dq_rows)
dq_path = DQ_DIR / 'gdelt_collection_report.csv'
dq_df.to_csv(dq_path, index=False)
print(f'Data quality report saved: {dq_path}')

## Manual Approval Gate

**Before approving, please verify:**
1. Sample titles above are genuinely about US labor markets
2. Monthly volume is reasonable (spikes during layoff waves are expected)
3. Top domains are credible news sources
4. No query group has 0 articles across all months
5. COVID months (2020-03 to 2020-05) show layoff/unemployment spikes

If the results look noisy or irrelevant, adjust `configs/gdelt_queries.yaml` and rerun.
Only approve once you are satisfied with data relevance.

In [ ]:
APPROVE_THIS_STEP = False

if not APPROVE_THIS_STEP:
    raise RuntimeError(
        'STOP: Inspect the diagnostics above. '
        'If everything is correct, change APPROVE_THIS_STEP=True and rerun this cell.'
    )

In [ ]:
import datetime, json

approval = {
    'approved': True,
    'notebook': '02_collect_raw_gdelt.ipynb',
    'approved_at': datetime.datetime.utcnow().isoformat(),
    'input_artifacts': [],
    'output_artifacts': [str(raw_path), str(dq_path)],
    'row_counts': {'news_raw': int(len(news_df))},
    'warnings': [],
    'notes': ''
}

approval_path = APPROVALS_DIR / '02_raw_gdelt_approved.json'
with open(approval_path, 'w') as f:
    json.dump(approval, f, indent=2)

print(f'Approval saved: {approval_path}')
print('Notebook 02 complete. Proceed to 03_audit_macro_vintage.ipynb')